# Data Preparation

## Import Libraries and Set Global Settings

In [ ]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

## Data Selection

### UK Dataset

In [ ]:
# Extract all CSV files from the path given
files = glob.glob("../data/raw/100,000 UK Used Car Data set/*.csv")

# Initialize dataframe list for each make 
dfs_uk = []

# Iterate through all files
for f in files:
    # Read each CSV file
    temp_df = pd.read_csv(f)
    # Extract filename of the CSV file
    file_name = os.path.basename(f).split('.')[0]
    # Use file name to add a 'make' feature
    temp_df['make'] = file_name
    # Add country feature to the dataset
    temp_df['country'] = 'uk'
    # Append the temporary dataframe into the list
    dfs_uk.append(temp_df)

# Concatenate all 'make' dataframes 
df_uk = pd.concat(dfs_uk, ignore_index=True)
# Print dataframe shape
print(f"Shape: {df_uk.shape}")

# Drop tax columns
df_uk = df_uk.drop(columns=['tax', 'tax(£)'], errors='ignore')

### Andrei Dataset

In [ ]:
# Define the file path for the Andrei Novikov dataset
andrei_path = "../data/raw/Used Cars Dataset Andrei Novikov/cars.csv"
# Read the CSV file into a dataframe
df_an = pd.read_csv(andrei_path)
# Add country feature to the dataset
df_an['country'] = 'usa'
# Print the shape of the dataset
print(f"Andrei Dataset Shape: {df_an.shape}")

# Rename columns to maintain consistency with the UK dataset
df_an = df_an.rename(columns={"manufacturer": "make", "fuel_type": "fuelType"})

# Define a function to extract numerical engine size from strings
def extract_engine_size(engine_str):
    
    if pd.isna(engine_str): 
        return np.nan

    # Use regular expression to find engine size
    match = re.search(r'(\d+\.?\d*)\s*L', str(engine_str))
    
    # Check if a match was found before extracting the value
    if match:
        return float(match.group(1))
    else:
        return np.nan

# Define a function to parse and average mpg range values
def parse_mpg(mpg_str):
    if pd.isna(mpg_str): 
        return np.nan

    # Split the string by the hyphen to identify ranges
    parts = str(mpg_str).split('-')
    try:
        # Initialize an empty list to store converted floats
        values = []
        # Iterate through parts and convert each to a float
        for p in parts:
            values.append(float(p))

        # Return the mean of the values
        return sum(values) / len(values)
    except: 
        return np.nan

# Apply engine size extraction to the engine column
df_an['engineSize'] = df_an['engine'].apply(extract_engine_size)
# Apply MPG parsing to the mpg column
df_an['mpg'] = df_an['mpg'].apply(parse_mpg)

## Data Cleaning

### UK Dataset

In [ ]:
# Drop duplicates
df_uk.drop_duplicates(inplace=True)

# Remove future cars
df_uk = df_uk[df_uk['year'] <= 2020]

# Filter categorical features
cat_features = df_uk.select_dtypes(include=['object', 'category']).columns

# Strip and lowercase all categorical features
for col in cat_features:
    df_uk[col] = df_uk[col].astype(str).str.lower().str.strip()

# Initialize model to make mapper for CSVs with model/shortened make names to make names
make_mapper = {'cclass': 'mercedes-benz', 
               'merc': 'mercedes-benz', 
               'mercedes': 'mercedes-benz',
               'focus': 'ford',
               'vw': 'volkswagen'
              }

# User the make mapper to replace the model/shortened make name to make names 
df_uk['make'] = df_uk['make'].replace(make_mapper)

### Andrei Dataset

In [ ]:
# Strip and lowercase transmission feature
df_an['transmission'] = df_an['transmission'].astype(str).str.lower().str.strip()

# Use regex to categorise automatic transmissions
auto_mask = df_an['transmission'].str.contains('auto|a/t|speed|cvt|variable|stronic|tiptronic', na=False)
df_an.loc[auto_mask, 'transmission'] = 'automatic'

# Use regex to categorise semi-auto transmissions
semi_mask = df_an['transmission'].str.contains('semi|dct|dual|clutch|pdk|dsg', na=False)
df_an.loc[semi_mask, 'transmission'] = 'semi-auto'

# Use regex to categorise manual transmissions
manual_mask = df_an['transmission'].str.contains('manual|m/t', na=False)
df_an.loc[manual_mask, 'transmission'] = 'manual'

# Standardise any remaining transmissions to 'automatic'
valid_trans = ['Automatic', 'Manual', 'Semi-Auto']
df_an.loc[~df_an['transmission'].isin(valid_trans), 'transmission'] = 'automatic'

# Initialise fuel type mapper
fuel_mapping_an = {
    'gasoline': 'petrol', 
    'premium': 'petrol', 
    'diesel': 'diesel', 
    'hybrid': 'hybrid', 
    'electric': 'electric'
}

# Use the fuel mapper to standardise fuel types
df_an['fuelType'] = df_an['fuelType'].astype(str).str.lower().str.strip().replace(fuel_mapping_an)

# Group non-standard fuel types into 'other'
valid_fuels = ['petrol', 'diesel', 'hybrid', 'electric']
df_an.loc[~df_an['fuelType'].isin(valid_fuels), 'fuelType'] = 'other'

# Filter outliers for engine size, price, and mpg
df_an = df_an[(df_an['engineSize'] <= 6.0) & (df_an['engineSize'] >= 1.0)]
df_an = df_an[df_an['price'] <= 70000]
df_an = df_an[df_an['mpg'] < 100]

# Print unique values for verification
print(f"Andrei Transmissions Cleaned: {df_an['transmission'].unique()}")
print(f"Andrei Fuel Types Cleaned: {df_an['fuelType'].unique()}")

### 